<a href="https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/HeuristicBaseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing Dataset from FlyRankAI

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## Preparing Dataset

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# 1. DATA COLLECTION
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Data Loaded: {df.shape[0]} rows.")

# 2. EDA (Functions available for visual audits)
def plot_danger_zone(data):
    data['pos_bin'] = pd.qcut(data['avg_position'], q=4, duplicates='drop')
    decay_rates = data.groupby('pos_bin')['is_declining_label'].mean() * 100
    decay_rates.plot(kind='bar', color='crimson', figsize=(8, 4))
    plt.title('Traffic Decay Rate by Ranking Position Quartile')
    plt.ylabel('% Declining')
    plt.show()

Data Loaded: 30000 rows.


## Handling Missing Values

In [9]:
# 3. DATA CLEANING (Handling Missing Values)
# Imputing the ~25% missing text metrics with the dataset mean
df['word_count'] = df['word_count'].fillna(df['word_count'].mean())
df['char_count'] = df['char_count'].fillna(df['char_count'].mean())

# 4. TRANSFORMATION & PREPROCESSING
# Converting the categorical trend into a binary machine learning target
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

print(f"Missing values handled. Target base rate: {df['is_declining_label'].mean():.3f}")

Missing values handled. Target base rate: 0.542


## Feature Engineering & Data Splitting

In [10]:
# 5. FEATURE ENGINEERING & SELECTION
# Creating the non-linear interaction feature
df['staleness_pos_interaction'] = df['days_since_last_update'] * df['avg_position']

# Defining the strict, pre-window feature contract (excluding leaky metrics)
base_features = [
    'days_since_last_update', 'content_age_days', 'impressions_90d',
    'avg_position', 'ctr', 'word_count', 'char_count', 'staleness_pos_interaction'
]

X = df[base_features]
y = df['is_declining_label']
groups = df['client_id']

# 6. DATA SPLITTING (Honest Validation)
# Grouping by client_id ensures domains do not bleed across the train/test barrier
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

df_test = df.iloc[test_idx].copy()
print(f"Test split created with {len(df_test)} rows from unseen clients.")

Test split created with 7115 rows from unseen clients.


## Constructing Heuristic Baseline

In [11]:
# 7. HEURISTIC BASELINE (Evaluated strictly on the Test Split)
# Rule: Score pages older than 180 days weighted by their 90-day impressions
df_test['baseline_score'] = (df_test['days_since_last_update'] > 180).astype(int) * df_test['impressions_90d']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Calculate Precision@50
baseline_p50 = precision_at_k(df_test['baseline_score'], df_test['is_declining_label'], k=50)

print(f"Honest Heuristic Baseline Precision@50: {baseline_p50:.3f}")

# Inspect the top 3 recommendations from the baseline
cols_to_review = ['content_id', 'days_since_last_update', 'impressions_90d', 'is_declining_label']
print("\nTop 3 Baseline Picks:")
display(df_test.sort_values(by='baseline_score', ascending=False)[cols_to_review].head(3))

Honest Heuristic Baseline Precision@50: 0.640

Top 3 Baseline Picks:


,content_id,days_since_last_update,impressions_90d,is_declining_label
22872,content_e3ff1b093148,183,1408,1
11630,content_6226ee6adc91,183,545,1
3507,content_074ba6ead17b,183,533,1


In [12]:
# 2. Calculate Baseline Score
df['baseline_score'] = (df['days_since_last_update'] > 180).astype(int) * df['impressions_90d']

# 3. Generate Ranked Queue
baseline_queue = df.sort_values(by='baseline_score', ascending=False)
top_50 = baseline_queue.head(50)

# 4. Evaluate Precision@50
p50 = top_50['is_declining_label'].mean()
print(f"Heuristic Baseline Precision@50: {p50:.3f}")

# 5. Export for Review
os.makedirs("work/outputs", exist_ok=True)
baseline_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

Heuristic Baseline Precision@50: 0.680
